In [1]:
import json, os, glob, tqdm
from PIL import Image
import numpy as np
import subprocess

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

# Axis 2

In [ ]:
ax = 2
method_json = f"/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/ffhq_rotateSH_userstudy_axis{ax}.json"
samples = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/HDRI_sota_sj.json"

method = [f"hou21_rotate_axis={ax}", f"hou22_rotate_axis={ax}", f"iclight_rotate_512x512_map_centered_axis{ax}", f"ours_difareli_rotate_rot{ax}", f"ours_difareli++_oneshot_rotate_rot{ax}_tomax"]
os.makedirs("./vids/", exist_ok=True)

with open(method_json, 'r') as f:
    method_json = json.load(f)

with open(samples, 'r') as f:
    samples = json.load(f)['pair']

no_f1 = True

for sample_id, sample in tqdm.tqdm(samples.items()):
    src = sample['src']
    dst = sample['dst']
    for m in method:
        img_dir = method_json[m]['img_dir']
        n_frames = method_json[m]['n_frame']
        if "hou21" in m or "hou22" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))
        elif "iclight" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))
        elif "ours" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))
        
        if len(relit) == 0:
            # Create 0 images for mockup and match the number of frames
            os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
            for i in range(1, int(n_frames)):
                img = Image.new('RGB', (256, 256), color='black')
                img.save(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_{i:04d}.png')
            continue
        
        if no_f1:
            relit = relit[1:]
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        # Copy all images to a new folder
        for img in relit:
            i = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}/res_frame_{i:04d}.png')

        #NOTE: Generate video high compression with 24 fps
        cmd = f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}/res_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/res_{m}.mp4'
        try:
            subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError as e:
            print("An error occurred:", e)


    # Copy the ball image
    ball = sorted(glob.glob(f"/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis={ax}/valid/{sample_id}_src={src}_dst={dst}/n_step={n_frames}/ball/m_*.png"))
    assert len(ball) == int(n_frames)
    if no_f1:
        ball = ball[1:]
    os.makedirs(f'./vids/axis={ax}/{src}_{dst}/ball/', exist_ok=True)
    for img in ball:
        os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/ball/{img.split("/")[-1]}')

    cmd = f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/ball/m_%03d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/ball.mp4'
    try:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print("An error occurred:", e)

    # Concatenate all videos vertically following the method order
    video_list = []
    for m in method:
        if os.path.exists(f'./vids/axis={ax}/{src}_{dst}/res_{m}.mp4'):
            video_list.append(f'./vids/axis={ax}/{src}_{dst}/res_{m}.mp4')

    video_list.append(f'./vids/axis={ax}/{src}_{dst}/ball.mp4')
    num_videos = len(video_list)
    # Generate FFmpeg inputs and filter_complex dynamically
    input_files = " ".join(f"-i {path}" for path in video_list)
    filter_complex = "".join(f"[{i}:v:0]" for i in range(num_videos)) + f"hstack=inputs={num_videos}"

    # Output file
    output_file = f"./vids/axis={ax}/{src}_{dst}/res_all.mp4"

    # Construct the FFmpeg command
    command = f"ffmpeg {input_files} -filter_complex \"{filter_complex}\" -c:v libx264 -crf 17 -preset slow {output_file}"

    # Execute the command
    try:
        subprocess.run(command, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print("An error occurred:", e)

  5%|▌         | 1/20 [00:05<01:35,  5.01s/it]


KeyError: 'ball'